# Model 3 — EfficientNet-B2
## COPD X-Ray Classification

**What is EfficientNet?**
EfficientNet by Google Brain (2019). Beats ResNet with fewer parameters.

**The key idea — compound scaling:**
Most people make a network better by making it:
- deeper (more layers), OR
- wider (more filters), OR
- higher resolution (bigger input)

EfficientNet does **all three at once** in a balanced way using a "compound coefficient."
This gives the best accuracy per parameter.

**EfficientNet-B2** is the second variant (B0 is smallest, B7 is biggest).
We use pretrained ImageNet weights and swap the last layer → 3 classes.

---
## Setup — Install & Import

In [ ]:
%pip install torch torchvision pillow scikit-learn matplotlib -q

In [ ]:
import zipfile
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torchvision.transforms as T
from torchvision import models
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
from sklearn.utils.class_weight import compute_class_weight
from PIL import Image

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

---
## Step 1 — Load the Data

Read images from the zip file. Assign 3 class labels:
- **Emphysema** → class 0
- **Normal** → class 1
- **Other** → class 2 (Covid, Pneumonia, Tuberculosis all go here)

In [ ]:
\
ZIP = r"c:\Users\xia7m\Desktop\Ai engnniering project\archive (6).zip"

LABEL_MAP = {
    "Emphysema"           : "Emphysema",
    "Normal"              : "Normal",
    "Covid-19"            : "Other",
    "Pneumonia-Bacterial" : "Other",
    "Pneumonia-Viral"     : "Other",
    "Tuberculosis"        : "Other",
}
LABEL_TO_NUM = {"Emphysema": 0, "Normal": 1, "Other": 2}
NUM_TO_LABEL = {0: "Emphysema", 1: "Normal", 2: "Other"}

def build_data(split):
    rows = []
    with zipfile.ZipFile(ZIP) as z:
        for filepath in z.namelist():
            parts = filepath.split("/")
            if len(parts) != 4:
                continue
            _, s, folder, fname = parts
            if s != split or folder not in LABEL_MAP:
                continue
            if not fname.lower().endswith(".jpg"):
                continue
            rows.append({"path": filepath, "label": LABEL_MAP[folder]})
    return rows

train_data = build_data("train")
val_data   = build_data("val")
test_data  = build_data("test")

print("Train:", len(train_data), "images")
print("Val  :", len(val_data),   "images")
print("Test :", len(test_data),  "images")

Show how many images per class.

In [ ]:
from collections import Counter

for split_name, split in [("Train", train_data), ("Val", val_data), ("Test", test_data)]:
    counts = Counter(e["label"] for e in split)
    print(f"{split_name}: {dict(counts)}")

---
## Step 2 — Transforms

**Train:** resize + augmentation (flip, rotate, color jitter) + normalize

**Val/Test:** resize + normalize only (no augmentation — we want stable results)

In [ ]:
train_transform = T.Compose([
    T.Resize((224, 224)),
    T.RandomHorizontalFlip(),
    T.RandomRotation(10),
    T.ColorJitter(brightness=0.2, contrast=0.2),
    T.ToTensor(),
    T.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

eval_transform = T.Compose([
    T.Resize((224, 224)),
    T.ToTensor(),
    T.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

print("Transforms ready.")

---
## Step 3 — Dataset & DataLoaders

Create a PyTorch Dataset that reads images from the zip file and returns
a (tensor, label) pair. Then wrap in DataLoaders for batching.

In [ ]:
class XRayDataset(Dataset):
    def __init__(self, data, zip_path, transform):
        self.data      = data
        self.zip_path  = zip_path
        self.transform = transform

    def __len__(self):
        return len(self.data)

    def __getitem__(self, i):
        entry = self.data[i]
        with zipfile.ZipFile(self.zip_path) as z:
            with z.open(entry["path"]) as f:
                img = Image.open(f).convert("RGB"); img.load()
        img   = self.transform(img)
        label = torch.tensor(LABEL_TO_NUM[entry["label"]])
        return img, label

BATCH_SIZE = 32

train_loader = DataLoader(XRayDataset(train_data, ZIP, train_transform), batch_size=BATCH_SIZE, shuffle=True,  num_workers=0)
val_loader   = DataLoader(XRayDataset(val_data,   ZIP, eval_transform),  batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
test_loader  = DataLoader(XRayDataset(test_data,  ZIP, eval_transform),  batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

imgs, lbls = next(iter(train_loader))
print("Batch shape:", imgs.shape, "| Labels shape:", lbls.shape)

---
## Step 4 — Class Weights

Some classes have fewer images than others. Without correction, the model
learns to ignore them. We fix this by giving higher loss penalty to minority classes.

We compute weights and pass them to the loss function (CrossEntropyLoss).

In [ ]:
all_labels    = [LABEL_TO_NUM[e["label"]] for e in train_data]
weights       = compute_class_weight("balanced", classes=np.array([0,1,2]), y=all_labels)
class_weights = torch.tensor(weights, dtype=torch.float32).to(device)

print("Class weights:")
for i, w in enumerate(class_weights.tolist()):
    print(f"  {NUM_TO_LABEL[i]:<12}: {w:.3f}")

---
## Step 5 — Training Loop

We define one function to train for one epoch and one to evaluate.
Then we loop 10 epochs, saving the best model each time.

In [ ]:
def train_one_epoch(model, loader, criterion, optimizer):
    model.train()
    total_loss, correct, total = 0, 0, 0
    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)
        outputs = model(images)
        loss    = criterion(outputs, labels)
        optimizer.zero_grad(); loss.backward(); optimizer.step()
        correct    += (outputs.argmax(1) == labels).sum().item()
        total      += labels.size(0)
        total_loss += loss.item()
    return total_loss / len(loader), correct / total * 100

def evaluate(model, loader, criterion):
    model.eval()
    total_loss, correct, total = 0, 0, 0
    with torch.no_grad():
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss    = criterion(outputs, labels)
            correct    += (outputs.argmax(1) == labels).sum().item()
            total      += labels.size(0)
            total_loss += loss.item()
    return total_loss / len(loader), correct / total * 100

---
## Step 6 — Load EfficientNet-B2

The classifier in EfficientNet is `model.classifier[1]` (a linear layer).
We replace it to output 3 classes.

In [ ]:
model = models.efficientnet_b2(weights="IMAGENET1K_V1")

# Freeze all layers
for param in model.parameters():
    param.requires_grad = False

# Replace the last linear layer
in_features = model.classifier[1].in_features
model.classifier[1] = nn.Linear(in_features, 3)

model = model.to(device)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total     = sum(p.numel() for p in model.parameters())
print(f"EfficientNet-B2 — total: {total:,} | trainable: {trainable:,}")

In [ ]:
EPOCHS    = 10
criterion = nn.CrossEntropyLoss(weight=class_weights)
optimizer = torch.optim.Adam(model.classifier.parameters(), lr=0.0001)

best_val_acc = 0
save_path    = "EfficientNetB2_best.pth"

for epoch in range(1, EPOCHS + 1):
    train_loss, train_acc = train_one_epoch(model, train_loader, criterion, optimizer)
    val_loss,   val_acc   = evaluate(model, val_loader, criterion)
    saved = ""
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(model.state_dict(), save_path)
        saved = " ← saved"
    print(f"Epoch {epoch:02d}/{EPOCHS}  train={train_acc:.1f}%  val={val_acc:.1f}%{saved}")

print(f"\nBest val accuracy: {best_val_acc:.1f}%")
model.load_state_dict(torch.load(save_path, map_location=device))

---
## Step 7 — Evaluate on Test Set

Load the best saved checkpoint and run it on the test set.

In [ ]:
test_criterion = nn.CrossEntropyLoss(weight=class_weights)
_, test_acc = evaluate(model, test_loader, test_criterion)
print(f"Test accuracy: {test_acc:.1f}%")

### Confusion Matrix

In [ ]:
model.eval()
all_preds, all_true = [], []

with torch.no_grad():
    for images, labels in test_loader:
        preds = model(images.to(device)).argmax(1).cpu().tolist()
        all_preds += preds
        all_true  += labels.tolist()

cm = confusion_matrix(all_true, all_preds)
labels = ["Emphysema", "Normal", "Other"]

plt.figure(figsize=(6, 5))
plt.imshow(cm, cmap="Blues")
plt.colorbar()
plt.xticks([0,1,2], labels, rotation=15)
plt.yticks([0,1,2], labels)
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title("Confusion Matrix")

for i in range(3):
    for j in range(3):
        plt.text(j, i, cm[i, j], ha="center", va="center",
                 color="white" if cm[i, j] > cm.max()/2 else "black",
                 fontsize=13, fontweight="bold")

plt.tight_layout()
plt.show()

### Classification Report (Precision, Recall, F1 per class)

In [ ]:
print(classification_report(all_true, all_preds, target_names=["Emphysema","Normal","Other"]))

---
## Step 8 — Save the Model

In [ ]:
import os
os.makedirs("saved_models", exist_ok=True)
torch.save(model.state_dict(), "saved_models/EfficientNetB2.pth")
print("Model saved to saved_models/EfficientNetB2.pth")